# Simple LLM Workflow

This notebook implements the simplest possible LangGraph workflow: a graph with a single node whose entire job is to make one call to a large language model and hand back the result.

A LangGraph workflow is built around three pieces: a **state** (a typed structure, here a `TypedDict`, that holds every value the workflow reads and writes), one or more **nodes** (plain Python functions that take the current state and return an updated state), and **edges** (the wiring that says which node runs after which, starting from a fixed `START` marker and ending at a fixed `END` marker).

In a "simple LLM workflow," an LLM call gets wrapped into a graph node by writing a function that: reads its input (a question, a prompt, or any other value) out of the incoming state; passes that input to an LLM client such as `ChatOpenAI`; takes the model's response and writes it into a new key on the state; and returns the updated state. That function is then registered on a `StateGraph` with `add_node`, and edges connect `START` to it and it to `END`. Compiling the graph produces a `workflow` object that can be run with `.invoke(...)` against an initial state dictionary.

State is what carries data through the graph: rather than passing arguments and return values between function calls directly, every node reads from and writes to the same shared dictionary, keyed by fields declared in the state schema. For a one-node graph this looks like overkill — a plain function call would do the same job with less code — but it is the same mechanism that scales to workflows with many steps, where each node consumes fields produced by an earlier node and produces fields consumed by a later one.

The reason to route even a single LLM call through LangGraph rather than calling the LLM directly is structural, not functional for this notebook alone: it establishes a state schema and a node/edge graph that can be extended with additional nodes (validation, retries, branching, multiple LLM calls in sequence) without changing how the workflow is invoked or how its result is read. This notebook is the minimal case; later notebooks in this folder build on it by chaining multiple nodes together into multi-step LLM workflows.

In [9]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv
import os


### Imports

This cell brings in every dependency the workflow needs:

- `StateGraph`, `START`, `END` from `langgraph.graph` — the core building blocks for defining a graph. `StateGraph` is the class used to declare nodes and edges over a shared state object; `START` and `END` are the fixed entry and exit markers used when wiring edges.
- `ChatOpenAI` from `langchain_openai` — a LangChain chat model wrapper around OpenAI's chat completion API. This is the LLM client the graph node will call.
- `TypedDict` from `typing` — used to define the state schema as a typed dictionary, so LangGraph knows what keys the state carries and can type-check access to them.
- `load_dotenv` from `dotenv` and `os` — used to load environment variables (such as the API key) from a `.env` file into the process environment, then read them with `os.getenv`.

No graph, model, or state is created yet — this cell only prepares the tools used by the rest of the notebook.

In [21]:
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

### Loading credentials from the environment

`load_dotenv()` reads a local `.env` file (if present) and injects its key-value pairs into the process environment. `os.getenv("OPENAI_API_KEY")` then retrieves the `OPENAI_API_KEY` value from that environment so it can be passed to the model client explicitly in the next cell.

Keeping the key in an environment variable rather than hardcoding it in the notebook means the value is never written into the notebook's own source, and the same code works across machines as long as each one has its own `.env` file or environment variable set. No key value is displayed or logged here.

In [22]:
model = ChatOpenAI(api_key=api_key)

### Instantiating the LLM client

`model = ChatOpenAI(api_key=api_key)` constructs a LangChain chat model bound to the OpenAI provider, authenticated with the key loaded in the previous cell. No `model` name is passed explicitly here, so `ChatOpenAI` falls back to its own default chat model.

This `model` object is a standalone LangChain runnable — it can be called directly with `.invoke(...)` outside of any graph, as later cells in this notebook demonstrate. Inside the graph, this same object is what the node function will call to actually generate a response; the graph itself does not talk to OpenAI directly, it delegates that work to `model`.

In [23]:
# create a state

class LLMState(TypedDict):

    question: str
    answer: str

### Defining the state schema

`LLMState` is a `TypedDict` with two keys:

- `question` — the input the workflow is given, a string holding the user's question.
- `answer` — the output the workflow produces, a string holding the LLM's response.

This dictionary shape is the single object that flows through the entire graph. Every node receives the current state and returns an updated state, so the schema declared here is the contract every node in the graph must respect: any node reading `question` or writing `answer` must do so through keys defined in this schema. With only one node in this workflow, `LLMState` simply describes "input in, output out," but the same pattern scales to graphs with many nodes, each reading and writing different keys of a shared, larger state.

In [24]:
def llm_qa(state: LLMState) -> LLMState:

    # extract the question from state
    question = state['question']

    # form a prompt
    prompt = f'Answer the following question {question}'

    # ask that question to the LLM
    answer = model.invoke(prompt).content

    # update the answer in the state
    state['answer'] = answer

    return state

### The node function

`llm_qa(state: LLMState) -> LLMState` is the single node this graph will run. It follows the standard LangGraph node contract: it takes the current state as its only argument and returns a (possibly updated) state of the same shape.

Inside the function:

1. `question = state['question']` reads the input out of the incoming state.
2. `prompt = f'Answer the following question {question}'` builds a plain-text prompt string embedding that question.
3. `answer = model.invoke(prompt).content` sends the prompt to the `ChatOpenAI` client. `.invoke(...)` returns a LangChain message object (an `AIMessage`), and `.content` extracts the plain text of the reply.
4. `state['answer'] = answer` writes the result back into the state dictionary under the `answer` key declared in `LLMState`.
5. The function returns the mutated `state`, which LangGraph merges into the graph's running state.

This function is not called directly anywhere in this cell — it is only defined here. The next cell registers it as a node so the graph can invoke it during execution.

In [25]:
# create our graph
graph = StateGraph(LLMState)

# add nodes
graph.add_node('llm_qa', llm_qa)

# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile
workflow = graph.compile()

### Building and compiling the graph

This cell assembles the actual `StateGraph`:

- `graph = StateGraph(LLMState)` creates a new graph parameterized by the `LLMState` schema, so LangGraph knows the shape of the state that will be passed between nodes.
- `graph.add_node('llm_qa', llm_qa)` registers the `llm_qa` function defined above as a node named `'llm_qa'`. The name is what edges will refer to.
- `graph.add_edge(START, 'llm_qa')` wires the graph's entry point directly to the `llm_qa` node — execution begins by invoking this node.
- `graph.add_edge('llm_qa', END)` wires `llm_qa`'s output straight to the graph's terminal marker — once the node returns, the graph run is finished.
- `workflow = graph.compile()` finalizes the graph definition into an executable `workflow` object.

Because there is exactly one node between `START` and `END`, this is the simplest possible LangGraph shape: a linear pipeline of length one. It still goes through the full graph API — state schema, node registration, explicit edges, compilation — which is what makes it straightforward to extend into a multi-node sequential workflow later.

In [26]:
# execute
intial_state = {'question': 'How far is moon from the earth?'}
final_state = workflow.invoke(intial_state)

print(final_state['answer'])

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************RhoA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

### Running the workflow

`intial_state = {'question': 'How far is moon from the earth?'}` builds the initial state dictionary matching the `LLMState` schema — only `question` needs to be supplied; `answer` is populated by the graph as it runs.

`final_state = workflow.invoke(intial_state)` executes the compiled graph starting from `START`, running the `llm_qa` node with `intial_state`, and returning the state after the node has updated it — this is the same `state` dictionary the `llm_qa` function returned, now containing both `question` and `answer`.

`print(final_state['answer'])` reads the LLM's response out of the returned state by key, exactly as it was written inside the node. The state dictionary is the only channel through which data enters and leaves the graph.

In [ ]:
model.invoke('How far is moon from the earth? and sun and saturn')

AIMessage(content='The average distance from the Earth to the Moon is about 384,400 kilometers (238,855 miles).', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 15, 'total_tokens': 37, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-C80VX9FswUz3L7ndBLBIgneB2430v', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--274bf984-94cc-4b22-9659-a1351b6b0685-0', usage_metadata={'input_tokens': 15, 'output_tokens': 22, 'total_tokens': 37, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

### Calling the LLM client directly, outside the graph

`model.invoke('How far is moon from the earth? and sun and saturn')` bypasses the graph entirely and calls the `ChatOpenAI` client that was created earlier with a raw string prompt. The returned object is the full LangChain `AIMessage`, not just its `.content` text — its output shown here includes the `content`, `response_metadata` (model name, finish reason, token usage), and `usage_metadata`.

This cell is a side-by-side contrast with the graph-based approach used above: the same `model.invoke(...)` call is what happens inside the `llm_qa` node, but here it is run standalone, with no state, no node, and no graph wrapping it. It illustrates that the graph does not change what the LLM call does — it only structures how that call is invoked, tracked, and connected to other steps.

In [19]:
from openai import OpenAI

client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY")
)

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "user", "content": "Say hello"}
    ]
)

print(response.choices[0].message.content)

Hello! How can I assist you today?


### Calling the raw OpenAI SDK, without LangChain or LangGraph

This final cell drops down a further layer, using the `OpenAI` client from the `openai` package directly instead of the LangChain `ChatOpenAI` wrapper:

- `client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))` builds a raw OpenAI SDK client, reading the same environment variable used earlier.
- `client.chat.completions.create(model="gpt-4.1-mini", messages=[{"role": "user", "content": "Say hello"}])` issues a chat completion request in the OpenAI SDK's native format — a `model` name and a `messages` list of role/content dictionaries, rather than a single prompt string.
- `response.choices[0].message.content` extracts the text of the first completion choice.

This cell has no connection to the `LLMState` graph built earlier; it exists to show the lowest-level API that both `ChatOpenAI` and the graph node ultimately sit on top of. Comparing it with the `model.invoke(...)` calls above shows what LangChain and LangGraph abstract away: message formatting, response parsing, and — once wrapped in a graph node — orchestration across multiple steps.

### Summary

This notebook builds the smallest possible LangGraph workflow: one state schema (`LLMState`), one node (`llm_qa`) that wraps a single call to an LLM, and a graph that connects `START` to that node and that node to `END`. The pattern established here is:

1. Define a state schema describing the data the workflow reads and writes.
2. Write a node function that takes that state, does work (here, one LLM call), and returns an updated state.
3. Register the node on a `StateGraph` and wire edges between `START`, the node, and `END`.
4. Compile the graph into a runnable `workflow` and invoke it with an initial state.

Because state is a plain dictionary shared across nodes rather than a single input/output pair, this same structure extends directly to workflows with more than one node — later notebooks in this folder (prompt chaining) add additional keys to the state and additional nodes between `START` and `END`, each performing one step of a multi-step LLM pipeline and passing its output forward through the shared state to the next node. The single-node graph in this notebook is the base case of that pattern.